In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import paths   # input/output locations: set SN2024VJM_DATA / SN2024VJM_EXTERNAL (see README)
from pathlib import Path
PATH_SPECTRA = paths.SPECTRA

from astropy.io import ascii
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.time import Time

%matplotlib widget

In [ ]:
a = ascii.read(PATH_SPECTRA / 'IR/SN2024vjm_0928_merged_tellcor.txt')
b = ascii.read(PATH_SPECTRA / 'IR/SN2024vjm_1010_merged_tellcor.txt')
e = ascii.read(PATH_SPECTRA / 'IR/SN2024vjm_1021_merged_tellcor.txt')
d = ascii.read(PATH_SPECTRA / 'TOO_StellarMerger_SCI_SLIT_FLUX_IDP_NIR.ascii')
jwst = ascii.read(PATH_SPECTRA / 'SN2024vjm_JWST_NIRSpec_60585.083.txt')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.ndimage import median_filter
from scipy.ndimage import gaussian_filter1d



wavelength = d['WAVE']*10
flux = d['FLUX']/np.nanmedian(d['FLUX'])

# remove these wavelengths from the spectrum: [[13450,14200],[18100,19356]]
mask = np.ones_like(flux, dtype=bool)
mask[(wavelength > 13450) & (wavelength < 14200)] = False
mask[(wavelength > 18100) & (wavelength < 19356)] = False
wavelength = wavelength[mask]
flux = flux[mask]


# Parameters for sigma clipping
window = 11  # must be odd; use a larger window if features are broad
sigma = 3    # threshold for clipping

# Smooth flux for comparison (you could also use a running median)
smoothed = median_filter(flux, size=window)

# Estimate noise (robust std using MAD)
residual = flux - smoothed
mad = np.median(np.abs(residual - np.median(residual)))
std_est = 1.4826 * mad  # convert MAD to std for Gaussian dist

# Sigma clipping
clipped_flux = np.copy(flux)
mask = np.abs(residual) > sigma * std_est
clipped_flux[mask] = smoothed[mask]
# clipped_flux = smooth_spectrum(clipped_flux, window=21, polyorder=11)

# Plot result
plt.figure(figsize=(10, 5))
plt.plot(wavelength, flux, color='gray', label='Original', alpha=0.6)
plt.plot(wavelength, clipped_flux, color='blue', lw=0.8, label='Sigma-clipped')
plt.xlabel(r'Wavelength [$\AA$]')
plt.ylabel('Flux [arbitrary units]')
plt.title('Sigma Clipping (σ = {})'.format(sigma))
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# save the clipped spectrum
# from astropy.io import ascii
# ascii.write({'WAVE': wavelength, 'FLUX': clipped_flux}, 'SN2024vjm_clipped_x-shooter_spectrum.txt', overwrite=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.ndimage import median_filter

z = 0.0028

# --- Define sigma clipping function ---
def sigma_clip_flux(flux, window=11, sigma=3):
    if window % 2 == 0:
        raise ValueError("Window size must be odd.")
    smoothed = median_filter(flux, size=window)
    residual = flux - smoothed
    mad = np.median(np.abs(residual - np.median(residual)))
    std_est = 1.4826 * mad
    clipped_flux = np.copy(flux)
    mask = np.abs(residual) > sigma * std_est
    clipped_flux[mask] = smoothed[mask]
    return clipped_flux, mask

# --- Define binning function ---
def bin_spectrum(wavelength, flux, bin_factor):
    """
    Rebins the spectrum by averaging over bin_factor points.
    """
    n_bins = len(flux) // bin_factor
    binned_wavelength = np.mean(wavelength[:n_bins * bin_factor].reshape(-1, bin_factor), axis=1)
    binned_flux = np.mean(flux[:n_bins * bin_factor].reshape(-1, bin_factor), axis=1)
    return binned_wavelength, binned_flux

# --- Load data ---
# d = fits.getdata('your_file.fits')
wavelength = d['WAVE'] * 10
flux = d['FLUX'] / np.nanmedian(d['FLUX'])

# --- Remove unwanted wavelength ranges ---
mask = np.ones_like(flux, dtype=bool)
mask[(wavelength > 13450) & (wavelength < 14200)] = False
mask[(wavelength > 18100) & (wavelength < 19356)] = False
wavelength = wavelength[mask]
flux = flux[mask]

# --- Iterative sigma clipping ---
max_iterations = 10
window = 21
sigma = 5
clipped_flux = np.copy(flux)
total_clipped_points = 0

for iteration in range(max_iterations):
    clipped_flux_new, mask = sigma_clip_flux(clipped_flux, window=window, sigma=sigma)
    n_clipped = np.sum(mask)
    print(f"Iteration {iteration + 1}: clipped {n_clipped} points")
    if n_clipped == 0:
        print("Converged: no more points clipped.")
        break
    total_clipped_points += n_clipped
    clipped_flux = clipped_flux_new

print(f"Total clipped points after {iteration + 1} iterations: {total_clipped_points}")

# --- Bin the spectrum ---
bin_factor = 5  # Change this as desired
binned_wavelength, binned_flux = bin_spectrum(wavelength, clipped_flux, bin_factor)
print(f"Binned spectrum by factor of {bin_factor}. Original length: {len(flux)}, Binned length: {len(binned_flux)}")

# --- Plot result ---
# plt.figure(figsize=(10, 5))
fig, ax = plt.subplots(figsize=(10, 5))
plt.plot(wavelength, flux, color='gray', label='X-shooter 1709 Original', alpha=0.5)
plt.plot(wavelength, clipped_flux, color='blue', lw=0.8, alpha=0.5)
plt.plot(binned_wavelength, binned_flux, color='blue', lw=0.8, label='X-shooter 1709 Sigma-clipped')
# plt.plot(binned_wavelength, binned_flux, color='red', lw=1.2, label=f'Binned x{bin_factor}')

#plot the other spectra
plt.plot(a['col1']*10000,a['col2']/np.nanmedian(a['col2'])+2, color='green',alpha=0.5, lw=0.8,label='SOAR 2809')
# plt.plot(jwst['col1']*10000,jwst['col2']/np.nanmedian(jwst['col2']), color='red',alpha=1, lw=0.8,label='JWST 0210')


plt.xlabel('Wavelength [nm]')
plt.ylabel('Flux [arbitrary units]')
# plt.title(f'Iterative Sigma Clipping + Binning (σ = {sigma}, window = {window}, bin x{bin_factor})')

elements = {'Co': 'I-III'}
low_w = '10000.0'
upp_w = '18000.0'
relative_intensity = 0.0
c = 299792.458  # speed of light in km/s

col_names = ["lam","ew","|ew/max|","cflux","lflux","lflux/max","nl","nu","ion","trans","type","lam_max","ewmax","lfluxmax","trans_max"]
lines =  pd.read_fwf(PATH_SPECTRA / 'def_2014_N1def_50d_lineid_ewfrac_0p0001_dlam_3500_10000.dat', header=None,index_col=None, skiprows=9,names=col_names)

def plot_ion(lines,ion,quntile,v,color='black'):
    """
    Plot vertical lines for a specific ion based on EW quantile.
    
    Parameters:
    ion (str): Ion to filter by (e.g., 'Co2').
    quntile (float): Quantile threshold for EW.
    v (float): Velocity shift in km/s.
    """
    filtered_lines = lines[lines['ion'] == ion].copy()
    ew_threshold = filtered_lines['ew'].quantile(quntile)
    strongest_lines = filtered_lines[filtered_lines['ew'] >= ew_threshold]
    
    for _, row in strongest_lines.iterrows():
        plt.axvline(x=(row['lam'] * (1 + z) * (1 - v / c)), color=color, linestyle='--', alpha=0.5, lw=1)
        plt.text((row['lam'] * (1 + z) * (1 - v / c)), 7, f"{ion}", rotation=90, fontsize=8, ha='center')

#set ylim
plt.ylim(0, 8)
plt.xlim(10000, 24000)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# --- Optional: save final binned spectrum ---
from astropy.io import ascii
ascii.write({'WAVE': wavelength, 'FLUX': clipped_flux}, str(PATH_SPECTRA / 'SN2024vjm_clipped_NIR_x-shooter_spectrum.txt'), overwrite=True)

In [ ]:
a = ascii.read(PATH_SPECTRA / 'IR/SN2024vjm_0928_merged_tellcor.txt')
b = ascii.read(PATH_SPECTRA / 'IR/SN2024vjm_1010_merged_tellcor.txt')
e = ascii.read(PATH_SPECTRA / 'IR/SN2024vjm_1021_merged_tellcor.txt')
jwst = ascii.read(PATH_SPECTRA / 'SN2024vjm_JWST_NIRSpec_60585.083.txt')

time_d = Time("2024-09-17T01:03:21.1465").mjd
time_a = Time("2024-09-28T00:06:35.427000").mjd
time_b = Time("2024-10-10T00:25:31.077000").mjd
time_e = Time("2024-10-22T01:01:26.277000").mjd

def clean_telluric_areas(wl,flux,error):
    mask = np.ones_like(flux, dtype=bool)
    # Wavelengths are in micrometers, not Angstroms
    mask[(wl > 1.345) & (wl < 1.420)] = False  # 13450-14200 Å = 1.345-1.420 μm
    mask[(wl > 1.810) & (wl < 1.936)] = False  # 18100-19356 Å = 1.810-1.936 μm
    mask[(wl<0.98)] = False
    wl = wl[mask]
    flux = flux[mask]
    error = error[mask]
    return wl*10000, flux,error

awl, aflux, a_error = clean_telluric_areas(a['col1'], a['col2'], a['col3'])
bwl, bflux, b_error = clean_telluric_areas(b['col1'], b['col2'], b['col3'])
ewl, eflux, e_error = clean_telluric_areas(e['col1'], e['col2'], e['col3'])

# Apply sigma clipping to the three SOAR spectra
print("Applying sigma clipping to SOAR spectra...")

# Parameters for sigma clipping
max_iterations = 10
window = 21
sigma = 5

# Process spectrum a
print(f"Processing spectrum a (original length: {len(aflux)})")
a_clipped_flux = np.copy(aflux / np.nanmedian(aflux))  # Normalize
total_clipped_a = 0

#Get rid of nan values in a:
awl = awl[~np.isnan(a_clipped_flux)]
a_clipped_flux = a_clipped_flux[~np.isnan(a_clipped_flux)]

for iteration in range(max_iterations):
    a_clipped_new, mask = sigma_clip_flux(a_clipped_flux, window=window, sigma=sigma)
    n_clipped = np.sum(mask)
    if n_clipped == 0:
        break
    total_clipped_a += n_clipped
    a_clipped_flux = a_clipped_new

print(f"Spectrum a: clipped {total_clipped_a} points in {iteration + 1} iterations")

# Process spectrum b
print(f"Processing spectrum b (original length: {len(bflux)})")
b_clipped_flux = np.copy(bflux / np.nanmedian(bflux))  # Normalize
total_clipped_b = 0

for iteration in range(max_iterations):
    b_clipped_new, mask = sigma_clip_flux(b_clipped_flux, window=window, sigma=sigma)
    n_clipped = np.sum(mask)
    if n_clipped == 0:
        break
    total_clipped_b += n_clipped
    b_clipped_flux = b_clipped_new

print(f"Spectrum b: clipped {total_clipped_b} points in {iteration + 1} iterations")

# Process spectrum e
print(f"Processing spectrum e (original length: {len(eflux)})")
e_clipped_flux = np.copy(eflux / np.nanmedian(eflux))  # Normalize
total_clipped_e = 0

for iteration in range(max_iterations):
    e_clipped_new, mask = sigma_clip_flux(e_clipped_flux, window=window, sigma=sigma)
    n_clipped = np.sum(mask)
    if n_clipped == 0:
        break
    total_clipped_e += n_clipped
    e_clipped_flux = e_clipped_new

print(f"Spectrum e: clipped {total_clipped_e} points in {iteration + 1} iterations")

print("\nSigma clipping completed for all three SOAR spectra.")

jwst_flux = jwst['col2'][jwst['col1'] < 2.4]
jwst_wl = jwst['col1'][jwst['col1'] < 2.4]*10000
#get rid of nans in the jwst data
jwst_wl = jwst_wl[~np.isnan(jwst_flux)]
jwst_flux = jwst_flux[~np.isnan(jwst_flux)]

#get rid of >22620 for wavelength and clipped_wavelength
binned_flux = binned_flux[binned_wavelength < 22620]
binned_wavelength = binned_wavelength[binned_wavelength < 22620]

#save the clipped spectra
from astropy.io import ascii
ascii.write({'WAVE': awl, 'FLUX': a_clipped_flux}, str(PATH_SPECTRA / 'SN2024vjm_0928_clipped_spectrum.txt'), overwrite=True)
ascii.write({'WAVE': bwl, 'FLUX': b_clipped_flux}, str(PATH_SPECTRA / 'SN2024vjm_1010_clipped_spectrum.txt'), overwrite=True)
ascii.write({'WAVE': ewl, 'FLUX': e_clipped_flux}, str(PATH_SPECTRA / 'SN2024vjm_1021_clipped_spectrum.txt'), overwrite=True)
ascii.write({'WAVE': jwst_wl, 'FLUX': jwst_flux}, str(PATH_SPECTRA / 'SN2024vjm_jwst_fixed_spectrum.txt'), overwrite=True)


In [ ]:
B_peak = 60572.18
z = 0.0028

plt.rcParams.update({'font.size': 14})

def plot_with_gaps(x, y, gap_threshold=100, **kwargs):
    """Plot with breaks where there are large gaps in wavelength"""
    # Find gaps larger than threshold
    dx = np.diff(x)
    gap_indices = np.where(dx > gap_threshold)[0]
    
    if len(gap_indices) == 0:
        # No gaps, plot normally
        plt.plot(x, y, **kwargs)
    else:
        # Plot segments between gaps
        start_idx = 0
        for gap_idx in gap_indices:
            end_idx = gap_idx + 1
            plt.plot(x[start_idx:end_idx], y[start_idx:end_idx], **kwargs)
            start_idx = end_idx
        # Plot final segment
        plt.plot(x[start_idx:], y[start_idx:], **kwargs)


fig,ax = plt.subplots(figsize=(14, 7))

plot_with_gaps(wavelength/(1+z), clipped_flux/np.nanmax(binned_flux)+0.3, color='black', lw=0.8, alpha=0.5)
plot_with_gaps(binned_wavelength/(1+z), binned_flux/np.nanmax(binned_flux)+0.3, color='black', lw=0.8, label='X-shooter 1709 Sigma-clipped')
plt.text(20200, 0.1+0.3, f"{(time_d - B_peak):.2f} d VLT/X-shooter", fontsize=12, color='black')

plot_with_gaps(awl/(1+z), a_clipped_flux/np.nanmax(a_clipped_flux) - 0.34, color='black', lw=0.8, label='SOAR 0928')
plt.text(20200, -0.15, f"+{(time_a - B_peak):.2f} d SOAR/TripleSpec", fontsize=12, color='black')

plot_with_gaps(jwst_wl/(1+z), jwst_flux/np.nanmax(jwst_flux)-1, color='black', alpha=1, lw=0.8, label='JWST 0210')
plt.text(20200, -0.45, f"+12 d JWST/NIRSpec", fontsize=12, color='black')

plot_with_gaps(bwl/(1+z), b_clipped_flux/np.nanmax(b_clipped_flux) - 1.5, color='black', lw=0.8, label='SOAR 1010')
plt.text(20200, -1.3, f"+{(time_b - B_peak):.2f} d SOAR/TripleSpec", fontsize=12, color='black')

plot_with_gaps(ewl/(1+z), e_clipped_flux/np.nanmax(e_clipped_flux) - 2, color='black', lw=0.8, label='SOAR 1021')
plt.text(20200, -1.8, f"+{(time_e - B_peak):.2f} d SOAR/TripleSpec", fontsize=12, color='black')


plt.fill_between([13400, 14200], -2.1, 0.9, color='gray', alpha=0.2)
plt.text(np.average([13400, 14200])-150, 0.8, r'$\oplus$', fontsize=20, color='black')
plt.fill_between([18000, 19356], -2.1, 0.9, color='gray', alpha=0.2)
plt.text(np.average([18000, 19356])-150, 0.8, r'$\oplus$', fontsize=20, color='black')
plt.fill_between([19900, 20200], -2.1, 0.9, color='gray', alpha=0.2)
plt.text(np.average([19900, 20200])-135, 0.8, r'$\oplus$', fontsize=20, color='black')

# 'MgII':  '#C3A384'
line = 10927
v=3000
plt.text(line-50, 0.68, 'Mg II', fontsize=12, color='black',rotation=90)
plt.fill_betweenx([-5, 10], line*(1+z)-220, line*(1+z)+200, color='#C3A384', alpha=0.2)
# plt.vlines([10830*(1-v/c)], -5, 10, color="#E70000", linestyle='--', alpha=0.5, lw=1)

line = 16900
#tilt the text 90 degrees
# plt.axvline(16907, color='red', linestyle='--', alpha=0.5, lw=1)
plt.text(line*(1+z)-85, 0.7, 'Fe II', fontsize=12, color='black', rotation=90)
plt.fill_betweenx([-5, 10], line*(1+z)-100, line*(1+z)+100, color='red', alpha=0.1)

plt.text(15572+400, 0.8, 'Co II', fontsize=12, color='black')
plt.fill_betweenx([-5, 10], 15572, line*(1+z)-100, color='#4169E1', alpha=0.1)
plt.text(line*(1+z)+300, 0.8, 'Co II', fontsize=12, color='black')
plt.fill_betweenx([-5, 10], line*(1+z)+100, 18000, color='#4169E1', alpha=0.1)
plt.text(21700+300, 0.8, 'Co II', fontsize=12, color='black')
plt.fill_betweenx([-5, 10], 20700, 23880, color='#4169E1', alpha=0.1)


# #He I 10830
# v=1500
# plt.vlines([10830*(1-v/c)], -5, 10, color='red')

# Set y and x limits
plt.ylim(-2.1, 0.9)  # +1 to leave space on top
ax.set_xlim(9000, 24000)

# # plot CO II lines 15759,16064,16361
# for line in [15759, 16064, 16361]:
#     plt.axvline(line+(1+z), color='red', linestyle='--', lw=1)

plt.xlabel(r'Rest-frame Wavelength ($\rm \AA$)')
plt.ylabel('Normalized Flux + Offset')
plt.yticks([])
# plt.legend(fontsize='small', loc='upper right')
plt.savefig('NIR_spectra_2024vjm.pdf',format='pdf', bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
# ===================== Co II P-Cygni joint fit (shared velocity) + PLOTS =====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from astropy.time import Time

# ----------------- user-configurable -----------------
COII_LINES = {
    "Co II 15759": 15759.0,
    "Co II 16064": 16064.0,
    "Co II 16361": 16361.0,
}
parent_fold = str(PATH_SPECTRA / 'IR') + '/'
WINDOW_A = 180.0                           # ± window around L0 for slicing (Å)
OUT_DIR = os.path.join(parent_fold, "coii_pcygni_jointfit")
c_kms = 299792.458
_K = np.sqrt(2.0*np.log(2.0))              # FWHM/2 = K * sigma

# --- phase reference ---
B_peak = 60572.18
JWST_MJD = 60585.083  # set None if unused

def phase_from_mjd(mjd, B_peak):
    try:
        return float(mjd) - float(B_peak)
    except Exception:
        return np.nan

# ----------------- per-line model pieces -----------------
def _line_model(lam, L0, a0, a1, A_em, mu_em, sig_em, A_abs, mu_abs, sig_abs):
    """
    Linear cont: a0 + a1*(lam-L0)
    Emission:    +A_em * exp(-0.5*((lam-mu_em)/sig_em)^2)
    Absorption:  -A_abs * exp(-0.5*((lam-mu_abs)/sig_abs)^2)
    """
    cont = a0 + a1*(lam - L0)
    em   = A_em  * np.exp(-0.5*((lam - mu_em)/np.maximum(sig_em,1e-6))**2)
    absb = A_abs * np.exp(-0.5*((lam - mu_abs)/np.maximum(sig_abs,1e-6))**2)
    return cont + em - absb

def _fwhm_velocity_kms(L0, sigma_A):
    FWHM_A = 2.0*_K*sigma_A
    return c_kms * FWHM_A / L0, FWHM_A

def _fwhm_velocity_err_kms(L0, pcov, sig_param_index_in_theta):
    try:
        var_sig = float(pcov[sig_param_index_in_theta, sig_param_index_in_theta])
        if not np.isfinite(var_sig) or var_sig < 0:
            return np.nan
        return (c_kms / L0) * (2.0*_K) * np.sqrt(var_sig)
    except Exception:
        return np.nan

# ----------------- JOINT fit for 3 lines in ONE spectrum -----------------
def fit_three_coii_lines_joint(wl_obs_A, flux, z, line_dict=COII_LINES, window_A=WINDOW_A):
    """
    Jointly fit the Co II lines in one spectrum with a COMMON absorption
    center velocity (v_exp_kms) and a COMMON absorption FWHM velocity (v_fwhm_kms).

    Parameter vector (theta) layout:
      [ v_exp_kms, v_fwhm_kms,
        (a0_1, a1_1, Aem_1, muem_1, sigem_1, Aabs_1),
        (a0_2, a1_2, Aem_2, muem_2, sigem_2, Aabs_2),
        (a0_3, a1_3, Aem_3, muem_3, sigem_3, Aabs_3) ]

    Notes
    -----
    - For each line i, the absorption center is fixed by the shared velocity:
        mu_abs_i = L0_i * (1 - v_exp_kms / c_kms)
    - For each line i, the absorption width (sigma_abs_i) is *derived* from the
      shared FWHM velocity via:
        sigma_abs_i [Å] = (v_fwhm_kms / c_kms) * L0_i / (2*K)
      so the FWHM in km/s is identical for all lines, while the Å-width scales with L0.
    """

    # ---- prep per-line windows (rest frame) ----
    wl_rest = wl_obs_A / (1.0 + z)
    arrays = []          # (name, L0, xx, yy_norm)
    for name, L0 in line_dict.items():
        m = (wl_rest >= L0 - window_A) & (wl_rest <= L0 + window_A)
        xx = wl_rest[m]
        yy = flux[m]
        if xx.size < 40:
            continue
        q75 = np.nanquantile(yy, 0.75)
        norm = q75 if np.isfinite(q75) and q75 != 0 else np.nanmedian(yy)
        if not np.isfinite(norm) or norm == 0:
            norm = 1.0
        arrays.append((name, L0, xx, yy / norm))

    if not arrays:
        return {"ok": False, "reason": "no_lines_with_coverage"}

    # concatenate data for curve_fit
    cat_x = np.concatenate([xx for (_, _, xx, _) in arrays])
    cat_y = np.concatenate([yy for (_, _, _, yy) in arrays])
    which_line = np.concatenate([np.full(xx.size, i, dtype=int)
                                 for i, (_, _, xx, _) in enumerate(arrays)])
    L0_per_line = np.array([L0 for (_, L0, _, _) in arrays], float)
    nL = len(arrays)

    # ---------- packing helpers ----------
    def unpack(theta):
        v_exp = theta[0]     # km/s (shared)
        v_fwhm = theta[1]    # km/s (shared)
        per_line = []
        base = 2
        for _ in range(nL):
            # (a0, a1, Aem, muem, sigem, Aabs)  -- NO per-line sigma_abs
            a0, a1, Aem, muem, sigem, Aabs = theta[base:base+6]
            per_line.append((a0, a1, Aem, muem, sigem, Aabs))
            base += 6
        return v_exp, v_fwhm, per_line

    def model_concat(lam_cat, *theta):
        v_exp, v_fwhm, per_line = unpack(theta)
        out = np.empty_like(lam_cat)
        for i in range(nL):
            idx = (which_line == i)
            if not np.any(idx):
                continue
            L0 = L0_per_line[i]
            a0, a1, Aem, muem, sigem, Aabs = per_line[i]
            mu_abs_i = L0 * (1.0 - v_exp/c_kms)
            # shared FWHM (km/s) → sigma_abs in Å for this L0
            sigma_abs_i = (v_fwhm / c_kms) * L0 / (2.0*_K)
            out[idx] = _line_model(lam_cat[idx], L0, a0, a1, Aem, muem, sigem,
                                   Aabs, mu_abs_i, sigma_abs_i)
        return out

    # ---------- initial guesses & bounds ----------
    p0 = [1500.0, 1200.0]          # v_exp_kms, v_fwhm_kms (both shared)
    low = [0.0,  400.0]            # velocity ≥ 0, modest min FWHM
    high = [10000.0, 4000.0]       # caps

    for _, L0, xx, yy in arrays:
        a0_0, a1_0 = float(np.nanmedian(yy)), 0.0
        Aem_0, muem_0, sigem_0 = 0.2, L0 + 0.15*0.5*window_A, 20.0
        Aabs_0 = 0.5
        p0 += [a0_0, a1_0, Aem_0, muem_0, sigem_0, Aabs_0]

        # Per-line bounds (emission & continuum & absorption depth only).
        low  += [0.0, -1.0/window_A, 0.0, L0 - window_A,   3.0,  0.0]
        high += [2.0,  1.0/window_A, 2.0, L0 + window_A, 120.0,  2.0]

    # mild central weighting per-point
    w = 1.0 / np.sqrt(1.0 + ((cat_x - L0_per_line[which_line])/(0.6*window_A))**2)

    # ---------- fit ----------
    try:
        popt, pcov = curve_fit(
            model_concat, cat_x, cat_y,
            p0=p0, bounds=(low, high),
            sigma=1.0/np.clip(w, 1e-3, None),
            absolute_sigma=False, maxfev=80000
        )
    except Exception as e:
        return {"ok": False, "reason": f"curve_fit_failed: {e}"}

    # unpack fitted parameters
    v_exp_kms, v_fwhm_kms, per_line = unpack(popt)
    v_exp_err_kms  = float(np.sqrt(pcov[0,0])) if (pcov is not None and np.isfinite(pcov[0,0])) else np.nan
    v_fwhm_err_kms = float(np.sqrt(pcov[1,1])) if (pcov is not None and np.isfinite(pcov[1,1])) else np.nan

    # assemble per-line rows
    rows = []
    for i, (name, L0, xx, yy) in enumerate(arrays):
        a0, a1, Aem, muem, sigem, Aabs = per_line[i]
        mu_abs = L0 * (1.0 - v_exp_kms/c_kms)
        sigma_abs_A = (v_fwhm_kms / c_kms) * L0 / (2.0*_K)
        FWHM_A = 2.0*_K*sigma_abs_A

        rows.append({
            "line": name, "L0_A": L0,
            "a0": a0, "a1": a1,
            "A_em": Aem, "mu_em_A": muem, "sig_em_A": sigem,
            "A_abs": Aabs, "mu_abs_A": mu_abs, "sig_abs_A": sigma_abs_A,
            "FWHM_A": FWHM_A,
            "v_center_kms": v_exp_kms,  "v_center_err_kms": v_exp_err_kms,   # shared
            "v_fwhm_kms":   v_fwhm_kms, "v_fwhm_err_kms":   v_fwhm_err_kms,  # shared
            "ok": True
        })

    return {"ok": True, "rows": rows}

# ----------------- helpers for plotting -----------------
def _prepare_spectra_list():
    """Build list of available spectra from already-loaded arrays."""
    spectra = []
    if 'awl' in globals() and 'a_clipped_flux' in globals() and 'time_a' in globals():
        spectra.append({"key": "SOAR a", "label": f"SOAR a  ({Time(time_a, format='mjd').isot[:10]})",
                        "wl_A": awl, "flux": a_clipped_flux, "phase": phase_from_mjd(time_a, B_peak)})
    if 'bwl' in globals() and 'b_clipped_flux' in globals() and 'time_b' in globals():
        spectra.append({"key": "SOAR b", "label": f"SOAR b  ({Time(time_b, format='mjd').isot[:10]})",
                        "wl_A": bwl, "flux": b_clipped_flux, "phase": phase_from_mjd(time_b, B_peak)})
    if 'ewl' in globals() and 'e_clipped_flux' in globals() and 'time_e' in globals():
        spectra.append({"key": "SOAR e", "label": f"SOAR e  ({Time(time_e, format='mjd').isot[:10]})",
                        "wl_A": ewl, "flux": e_clipped_flux, "phase": phase_from_mjd(time_e, B_peak)})
    if 'jwst_wl' in globals() and 'jwst_flux' in globals():
        spectra.append({"key": "JWST/NIRSpec", "label": "JWST/NIRSpec",
                        "wl_A": jwst_wl, "flux": jwst_flux,
                        "phase": phase_from_mjd(JWST_MJD, B_peak) if 'JWST_MJD' in globals() else np.nan})
    return spectra

def _flex_match_rows(df, spectrum_label, phase_target, phase_tol=2.0):
    """
    Return subset of df for this spectrum, being flexible about names:
    1) exact match on 'spectrum'
    2) substring match (spectrum_label contained in df['spectrum'])
    3) nearest phase within ±phase_tol days (then take that spectrum name)
    """
    rows = df[df["spectrum"] == spectrum_label]
    if not rows.empty:
        return rows
    m = df["spectrum"].astype(str).str.contains(spectrum_label, case=False, regex=False)
    rows = df[m]
    if not rows.empty:
        return rows
    if np.isfinite(phase_target):
        df_ok = df[np.isfinite(df["phase_days"])]
        if not df_ok.empty:
            i = (df_ok["phase_days"] - phase_target).abs().idxmin()
            if np.isfinite(df_ok.loc[i, "phase_days"]) and abs(df_ok.loc[i, "phase_days"] - phase_target) <= phase_tol:
                spec_name = df_ok.loc[i, "spectrum"]
                return df[df["spectrum"] == spec_name]
    return df.iloc[0:0]  # empty

# ----------------- batch over already-loaded spectra (fit + CSV) -----------------
def run_coii_jointfit_saved():
    """
    Uses already-loaded arrays (see _prepare_spectra_list).
    Saves one CSV of joint-fit results for each spectrum × line.
    """
    os.makedirs(OUT_DIR, exist_ok=True)
    spectra = _prepare_spectra_list()

    all_rows = []
    for sp in spectra:
        res = fit_three_coii_lines_joint(sp["wl_A"], sp["flux"], z, COII_LINES, WINDOW_A)
        if not res.get("ok", False):
            for ln, L0 in COII_LINES.items():
                all_rows.append({
                    "spectrum": sp["label"], "phase_days": sp["phase"],
                    "line": ln, "L0_A": L0, "ok": False, "reason": res.get("reason","fit_failed")
                })
            continue
        for row in res["rows"]:
            row.update({"spectrum": sp["label"], "phase_days": sp["phase"]})
            all_rows.append(row)

    df = pd.DataFrame(all_rows).sort_values(["phase_days","spectrum","line"]).reset_index(drop=True)
    out_csv = os.path.join(OUT_DIR, "coii_jointfit_results.csv")
    df.to_csv(out_csv, index=False)
    print("Saved CSV:", out_csv)
    return df, out_csv

# ----------------- PLOTTING: per-spectrum mosaics (3 panels) -----------------
def plot_coii_mosaics_from_df(df_coii, line_dict=COII_LINES, out_dir=OUT_DIR):
    """
    For each available spectrum in memory, build a 3-panel mosaic (one per line),
    overlaying the fitted model/components on a freshly normalized window.
    """
    os.makedirs(out_dir, exist_ok=True)
    spectra = _prepare_spectra_list()

    for sp in spectra:
        rows_sp = _flex_match_rows(df_coii, sp["label"], sp["phase"])
        if rows_sp.empty:
            print(f"[mosaic] No rows in df_coii for spectrum: {sp['label']}")
            continue

        # canvas
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), squeeze=False)
        axes = axes[0]

        for j, (line_name, L0) in enumerate(line_dict.items()):
            ax = axes[j]
            row_line = rows_sp[rows_sp["line"] == line_name]
            if row_line.empty:
                ax.text(0.5, 0.5, "no fit", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(line_name, fontsize=11)
                ax.set_xlabel(r"Rest λ (Å)"); ax.set_ylabel("Flux (norm)")
                continue
            r = row_line.iloc[0]

            # slice + normalize the observed spectrum exactly like in the fit
            wl_rest = sp["wl_A"] / (1.0 + z)
            m = (wl_rest >= L0 - WINDOW_A) & (wl_rest <= L0 + WINDOW_A)
            xs = wl_rest[m]; ys = sp["flux"][m]
            if xs.size < 5:
                ax.text(0.5, 0.5, "no coverage", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(line_name, fontsize=11)
                ax.set_xlabel(r"Rest λ (Å)"); ax.set_ylabel("Flux (norm)")
                continue
            q75 = np.nanquantile(ys, 0.75)
            norm = q75 if np.isfinite(q75) and q75 != 0 else np.nanmedian(ys)
            if not np.isfinite(norm) or norm == 0: norm = 1.0
            ysn = ys / norm

            # rebuild model/components from stored params
            a0, a1 = r["a0"], r["a1"]
            Aem, muem, sigem = r["A_em"], r["mu_em_A"], r["sig_em_A"]
            Aabs, sigabs = r["A_abs"], r["sig_abs_A"]
            # center is the shared velocity:
            v_center = r["v_center_kms"]
            mu_abs = L0 * (1.0 - v_center/c_kms)

            model = _line_model(xs, L0, a0, a1, Aem, muem, sigem, Aabs, mu_abs, sigabs)
            cont  = a0 + a1*(xs - L0)
            em    = Aem * np.exp(-0.5*((xs - muem)/np.maximum(sigem,1e-6))**2)
            ab    = Aabs * np.exp(-0.5*((xs - mu_abs)/np.maximum(sigabs,1e-6))**2)

            # FWHM guides
            lam_left  = mu_abs - _K*sigabs
            lam_right = mu_abs + _K*sigabs
            half_depth_level = np.nanmedian(cont) - 0.5*Aabs

            # draw
            ax.plot(xs, ysn, lw=1.0, alpha=0.9, label="data (norm)")
            ax.plot(xs, model, lw=2.0, label="fit")
            ax.plot(xs, cont, ls="--", alpha=0.8, label="continuum")
            ax.plot(xs, cont + em, ls="-.", alpha=0.8, label="emission comp")
            ax.plot(xs, cont - ab, ls=":",  alpha=0.8, label="absorption comp")
            ax.axvline(L0, color="k", ls=":", alpha=0.5)
            ax.axvline(mu_abs, color="r", ls="--", alpha=0.7, label="abs center")
            ax.axhline(half_depth_level, color="gray", ls="--", alpha=0.4, label="half depth")
            ax.axvline(lam_left,  color="b", ls="--", alpha=0.7, label="FWHM edges")
            ax.axvline(lam_right, color="b", ls="--", alpha=0.7)

            vtxt = f"v_center={r['v_center_kms']:.0f}±{r['v_center_err_kms']:.0f} km/s\n" \
                   f"v_FWHM={r['v_fwhm_kms']:.0f}±{r['v_fwhm_err_kms']:.0f} km/s"
            ax.text(0.02, 0.98, vtxt, transform=ax.transAxes, va="top", ha="left", fontsize=9,
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.65, lw=0))

            ax.set_title(line_name, fontsize=11)
            ax.set_xlabel(r"Rest λ (Å)")
            if j == 0:
                ax.set_ylabel("Flux (norm)")

        ttl = f"{sp['label']}  |  phase {sp['phase']:+.1f} d" if np.isfinite(sp["phase"]) else sp["label"]
        fig.suptitle(ttl, y=1.02, fontsize=13)
        # unique legend
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc="upper right", frameon=False)
        fig.tight_layout()
        out_png = os.path.join(out_dir, f"coii_mosaic_from_df_{sp['key'].replace('/','-')}.png")
        fig.savefig(out_png, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print("Saved mosaic:", out_png)

# ----------------- PLOTTING: velocities vs phase (center & FWHM) -----------------
def plot_coii_velocities_vs_phase(df_coii, out_dir=OUT_DIR,
                                  max_rel_err=0.7, max_abs_err=4000):
    """
    Single panel: plot v_center and v_FWHM vs phase for all lines.
    Colors by line, shapes indicate metric (center vs FWHM).
    """
    os.makedirs(out_dir, exist_ok=True)
    q = df_coii.copy()
    q = q[(q["ok"]) & np.isfinite(q["phase_days"])]

    if q.empty:
        print("No valid rows to plot.")
        return

    # prune crazy errors
    def _keep(v, e):
        return np.isfinite(v) & np.isfinite(e) & (e < max_abs_err) & (e < max_rel_err*np.abs(v).clip(min=1.0))
    m_center = _keep(q["v_center_kms"].values, q["v_center_err_kms"].values)
    m_fwhm   = _keep(q["v_fwhm_kms"].values,   q["v_fwhm_err_kms"].values)

    line_colors = {
        "Co II 15759": "#1f77b4",
        "Co II 16064": "#2ca02c",
        "Co II 16361": "#d62728",
    }

    fig, ax = plt.subplots(figsize=(8.5, 5.5))

    # plot v_center
    for ln in COII_LINES.keys():
        df_ln = q[(q["line"] == ln) & m_center].sort_values("phase_days")
        if df_ln.empty: continue
        ax.errorbar(df_ln["phase_days"], df_ln["v_center_kms"], yerr=df_ln["v_center_err_kms"],
                    fmt="o", ms=5, lw=1.0, capsize=3, color=line_colors.get(ln, "#9467bd"),
                    ecolor=line_colors.get(ln, "#9467bd"), label=f"{ln} (center)")

    # plot v_FWHM
    for ln in COII_LINES.keys():
        df_ln = q[(q["line"] == ln) & m_fwhm].sort_values("phase_days")
        if df_ln.empty: continue
        ax.errorbar(df_ln["phase_days"], df_ln["v_fwhm_kms"], yerr=df_ln["v_fwhm_err_kms"],
                    fmt="s", ms=5, lw=1.0, capsize=3, color=line_colors.get(ln, "#9467bd"),
                    ecolor=line_colors.get(ln, "#9467bd"), alpha=0.9, label=f"{ln} (FWHM)")

    ax.axhline(0, color="k", ls=":", lw=0.7)
    ax.set_xlabel("Phase (days)")
    ax.set_ylabel("Velocity (km/s)")
    ax.set_title("Co II velocities vs phase (shared-center fit)")
    ax.legend(ncol=2, frameon=False, fontsize=9)
    fig.tight_layout()
    out_png = os.path.join(out_dir, "coii_velocities_vs_phase.png")
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Saved summary:", out_png)

# ---- run: fit + save + plot ----
df_coii_joint, csv_coii_joint = run_coii_jointfit_saved()
print(f"Rows written: {len(df_coii_joint)}")

# mosaics per spectrum (3 panels each)
plot_coii_mosaics_from_df(df_coii_joint, COII_LINES, OUT_DIR)

# velocities vs phase
plot_coii_velocities_vs_phase(df_coii_joint, OUT_DIR)

In [ ]:
df_coii_joint

In [ ]:
from astropy import units as u

c = 2.998e18
jwst = ascii.read(PATH_SPECTRA / 'SN2024vjm_JWST_NIRSpec_60585.083.txt')

jwst_flux = jwst['col2'][jwst['col1']>2.4]
jwst_wl = jwst['col1'][jwst['col1']>2.4]*10000
jwst_flux =jwst_flux *1e3*1e-26*c/(jwst_wl)**2
#get rid of nans in the jwst data
jwst_wl = jwst_wl[~np.isnan(jwst_flux)]
jwst_flux = jwst_flux[~np.isnan(jwst_flux)]
np.trapz(jwst_flux,x=jwst_wl)

sphere = 4*np.pi*(7.92e6 * u.pc.to("cm"))**2
sphere*np.trapz(jwst_flux,x=jwst_wl)
